# 03 - Train the v3 text and audio student branches from scratch

Trains the two compact hard-label branches that feed the multimodal student. These are newly initialized models; no old experiment checkpoints are loaded.

Both branches save their random initialization separately as `random_init.*` diagnostics. These are **not** teacher epoch 0. Teacher epoch 0 is reserved for the original published/released teacher checkpoints in notebooks 01 and 02. Existing branch-output folders are never overwritten.

Notebook 03 is restart-safe: an exact text branch is reused; non-exact older audio attempts are archived, not deleted; new-format interrupted audio runs resume from `last_state.pt`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '0d6ac621be7da1db7c1d769d075d142d0a740fd8'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
IDIAP_REPO='https://github.com/idiap/bias_in_daic-woz.git'
IDIAP_COMMIT='de8f477619ce61a2a65640a0735aa8dfe659ac59'
IDIAP=Path('/content/idiap-bias_in_daic-woz')
if IDIAP.exists(): shutil.rmtree(IDIAP)
subprocess.run(['git','clone',IDIAP_REPO,str(IDIAP)],check=True)
subprocess.run(['git','-C',str(IDIAP),'checkout','--detach',IDIAP_COMMIT],check=True)

USSD=Path('/content/USSD-depression')
if USSD.exists(): shutil.rmtree(USSD)
subprocess.run(['git','clone','https://github.com/vijaysumaravi/USSD-depression.git',str(USSD)],check=True)
subprocess.run(['git','-C',str(USSD),'checkout','--detach','c3e68649153004ed2174878a1c54c716ad26cfd7'],check=True)

In [ ]:
from datetime import datetime, timezone

TEXT=RUN/'03_student_text'
AUDIO=RUN/'03_student_audio'
CACHE_POINTER=RUN/'02_audio_feature_cache.json'
assert CACHE_POINTER.is_file(),'Run notebook 02 first.'
cache_ptr=json.loads(CACHE_POINTER.read_text())
FEATURES=Path(cache_ptr['path'])
assert FEATURES.is_dir(),FEATURES
assert cache_ptr['train_participants']==107
assert cache_ptr['dev_participants']==34
assert cache_ptr['participant_440_excluded'] is True
assert cache_ptr['test_opened'] is False
assert cache_ptr['compare16_config_sha256']=='4baf8b75324db30e632a2935ab469058d54cd1c7b1806d5cc220dc7b124c454c'
print('Using validated shared audio feature cache:',FEATURES)

# Self-heal a stale Colab code checkout. This cell may be rerun even when the
# setup cell was executed before Git was patched.
def ensure_pinned_core():
    try:
        current=subprocess.check_output(
            ['git','-C',str(CODE),'rev-parse','HEAD'],text=True,
            stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        current=None

    if current!=CORE_COMMIT:
        print('Refreshing stale code checkout:',current,'->',CORE_COMMIT)
        if CODE.exists(): shutil.rmtree(CODE)
        CODE.mkdir(parents=True)
        subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
        subprocess.run(['git','-C',str(CODE),'remote','add','origin',
                        f'https://github.com/{REPO}.git'],check=True,env=env)
        subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],
                       check=True,env=env)
        subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],
                       check=True,env=env)

    current=subprocess.check_output(
        ['git','-C',str(CODE),'rev-parse','HEAD'],text=True
    ).strip()
    assert current==CORE_COMMIT,(current,CORE_COMMIT)

    help_run=subprocess.run(
        [sys.executable,'-m','scripts.students.participant_student_v3.pretrain_audio','--help'],
        cwd=CODE,text=True,capture_output=True
    )
    help_text=(help_run.stdout or '')+'\n'+(help_run.stderr or '')
    assert help_run.returncode==0,help_text
    assert '--exact-v3' in help_text and '--resume' in help_text,(
        'Pinned pretrain_audio CLI does not contain exact-v3 resume flags.\n'+help_text
    )
    print('Verified audio training core:',current)

ensure_pinned_core()

def text_exact_ok(out):
    try:
        d=json.loads((out/'metrics.json').read_text()); m=d['dev34']
        return (
            round(float(m['macro_f1']),4)==0.7850 and
            round(float(m['depressed_f1']),4)==0.7407 and
            round(float(m['auroc']),4)==0.8063 and
            m['confusion_matrix']==[[17,6],[1,10]]
        )
    except Exception:
        return False

def audio_exact_ok(out):
    try:
        d=json.loads((out/'metrics.json').read_text()); m=d['dev34']; p=d['protocol']
        complete=json.loads((out/'complete.json').read_text())
        return (
            complete.get('status')=='PASS' and
            p['batch_size']==20 and p['crop_seed']==1300 and
            p['requested_epochs']==100 and p['patience']==20 and
            p['completed_epochs']==71 and p['best_epoch']==51 and
            round(float(m['majority_vote']['macro_f1']),4)==0.6222 and
            round(float(m['majority_vote']['depressed_f1']),4)==0.4444 and
            round(float(m['auroc_soft_mean_probability']),4)==0.5257 and
            m['majority_vote']['confusion_matrix']==[[20,3],[7,4]]
        )
    except Exception:
        return False

def archive_attempt(path):
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
    dst=path.with_name(path.name+'_attempt_'+stamp)
    i=1
    while dst.exists():
        dst=path.with_name(path.name+'_attempt_'+stamp+f'_{i}')
        i+=1
    shutil.move(str(path),str(dst))
    print('Preserved previous attempt:',dst)
    return dst

# TEXT: reuse exact result; otherwise preserve and rebuild.
if TEXT.exists() and text_exact_ok(TEXT):
    print('REUSE: exact v3 text branch already present:',TEXT)
else:
    if TEXT.exists(): archive_attempt(TEXT)
    subprocess.run([sys.executable,'-m','scripts.students.participant_student_v3.pretrain_text',
                    '--daic-root',str(DAIC),'--idiap-source',str(IDIAP),
                    '--output',str(TEXT),'--seed','17','--eval-every','1'],cwd=CODE,check=True)

# AUDIO:
# - exact complete result -> reuse
# - interrupted new-format result with last_state.pt -> resume in place
# - old/mismatched completed attempt -> preserve, then start clean
if AUDIO.exists() and audio_exact_ok(AUDIO):
    print('REUSE: exact v3 audio branch already present:',AUDIO)
else:
    resume_ok=(
        AUDIO.exists() and
        (AUDIO/'last_state.pt').is_file() and
        not (AUDIO/'reproduction_failure.json').is_file()
    )
    if AUDIO.exists() and not resume_ok:
        archive_attempt(AUDIO)
        resume_ok=False

    cmd=[sys.executable,'-m','scripts.students.participant_student_v3.pretrain_audio',
         '--features',str(FEATURES),'--author-root',str(USSD),
         '--output',str(AUDIO),'--seed','1300','--batch-size','20',
         '--epochs','100','--patience','20','--lr','0.003',
         '--lr-factor-epochs','2','--lr-decay','0.9',
         '--exact-v3','--resume']
    print('AUDIO mode:', 'RESUME' if resume_ok else 'FRESH EXACT RUN')
    print('Audio command uses exact-v3 + resume-capable core.')
    proc=subprocess.run(cmd,cwd=CODE,text=True)
    if proc.returncode!=0:
        raise RuntimeError(
            f'Audio training exited with code {proc.returncode}. '
            'The real script error is printed immediately above; '
            'any last_state.pt produced before failure is preserved for resume.'
        )

In [ ]:
t=json.loads((TEXT/'metrics.json').read_text())
a=json.loads((AUDIO/'metrics.json').read_text())
tm=t['dev34']; am=a['dev34']
t0=t['random_init_dev34']; a0=a['random_init_dev34']

print('TEXT RANDOM-INIT DEV:',json.dumps(t0,indent=2))
print('TEXT BEST DEV:',json.dumps(tm,indent=2))
print('AUDIO RANDOM-INIT DEV:',json.dumps(a0,indent=2))
print('AUDIO BEST DEV:',json.dumps(am,indent=2))

assert t['protocol']['train_participants']==107 and t['protocol']['dev_participants']==34
assert a['protocol']['train_participants']==107 and a['protocol']['dev_participants']==34

# Zero-epoch audit models must exist but never participate in checkpoint selection.
for p in [TEXT/'random_init.pt',TEXT/'random_init_metrics.json',TEXT/'random_init_dev_predictions.csv',
          AUDIO/'random_init.pt',AUDIO/'random_init_metrics.json',AUDIO/'random_init_dev_predictions.csv']:
    assert p.is_file(),p
assert t['protocol']['random_init_saved'] is True and t['protocol']['random_init_selection_eligible'] is False
assert a['protocol']['random_init_saved'] is True and a['protocol']['random_init_selection_eligible'] is False

# Historical v3 text branch.
assert round(float(tm['macro_f1']),4)==0.7850
assert round(float(tm['depressed_f1']),4)==0.7407
assert round(float(tm['auroc']),4)==0.8063
assert tm['confusion_matrix']==[[17,6],[1,10]]

# Historical v3 audio recipe must be identical, not merely "good".
assert a['protocol']['batch_size']==20,a['protocol']
assert a['protocol']['crop_seed']==1300,a['protocol']
assert a['protocol']['requested_epochs']==100,a['protocol']
assert a['protocol']['patience']==20,a['protocol']
assert a['protocol']['completed_epochs']==71,a['protocol']
assert a['protocol']['best_epoch']==51,a['protocol']
assert json.loads((AUDIO/'complete.json').read_text())['status']=='PASS'
assert round(float(am['majority_vote']['macro_f1']),4)==0.6222
assert round(float(am['majority_vote']['depressed_f1']),4)==0.4444
assert round(float(am['auroc_soft_mean_probability']),4)==0.5257
assert am['majority_vote']['confusion_matrix']==[[20,3],[7,4]]

print('PASS: exact v3 student branches reproduced, with random-initialization diagnostics preserved separately.')